# 03 — Evaluation: all three conditions

Runs base / SFT-only / distilled students through the agentic harness at chain lengths 1, 3, 5 (the synthetic task set — `harness/tasks.py::load_tasks(chain_length)`, default `source="synthetic"`; see README's "Why synthetic tasks for multi-step eval"), plus the general-LM perplexity baseline. `load_tasks(1, source="glaive_test")` is available as a secondary, single-step-only, in-distribution data point but isn't part of the chain-length comparison itself.

Requires, before running this notebook:
- `notebooks/02_training.ipynb` to have produced all three conditions' checkpoints
- `python -m adbench.data.general_eval` to have produced `data/general_eval/wikitext2_sample.jsonl`

This notebook only orchestrates — all the actual logic (config resolution, the harness-run loop, the model_fn wiring, perplexity, metrics, output writing) lives in `adbench.evaluation.run_eval`/`.metrics`/`.perplexity` and is unit-tested locally (`tests/test_run_eval.py`, `tests/test_metrics.py`, `tests/test_perplexity.py`) without a GPU — only the actual checkpoint loading needs one.

In [29]:
# Public repo — no auth needed to clone. Skips re-cloning if this
# session's runtime already has the repo (e.g. you ran
# 00_setup_colab.ipynb earlier in this same session).
import os
import sys

# Reduces CUDA OOM from a single large allocation (e.g. loading the 14B
# teacher) by letting the allocator grow a segment incrementally instead of
# needing one big contiguous block upfront.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Nahla-Nabil/agentic-distillation-benchmark.git {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q -r requirements-colab.txt

# Put src/ on sys.path (for `import adbench` right here in this kernel)
# AND on PYTHONPATH (for `!python -m adbench...` subprocess calls in later
# cells, which inherit the environment but not this process's sys.path)
# instead of `pip install -e .` — an editable install registers itself via
# a .pth file that Python's site module only reads at interpreter startup,
# so `import adbench` fails with ModuleNotFoundError in this same
# still-running kernel until you restart it. Both of the below work
# immediately, no restart needed, on Colab or Kaggle.
src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')

In [30]:
# Re-run this any time (after I've pushed a fix) to sync this session's
# cloned repo to the latest on GitHub, without re-cloning or restarting.
!cd {REPO_DIR} && git checkout -- . && git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 8 (delta 6), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 3.51 KiB | 1.75 MiB/s, done.
From https://github.com/Nahla-Nabil/agentic-distillation-benchmark
   2fbb60c..013e9fa  master     -> origin/master
Updating 2fbb60c..013e9fa
Fast-forward
 notebooks/03_evaluate.ipynb | 208 +++++++++++++++++++++++++++++---------------
 1 file changed, 136 insertions(+), 72 deletions(-)


In [31]:
# Runs ALL THREE conditions x all three chain lengths + perplexity in one pass
# (equivalent to `python -m adbench.evaluation.run_eval` from the CLI):
from adbench.evaluation.run_eval import evaluate_condition, write_eval_outputs
from adbench.training.train import CONDITIONS, REPO_ROOT, load_experiment_config

experiment_config = load_experiment_config("configs/experiment.yaml")
models_config = load_experiment_config("configs/models.yaml")
chain_lengths = experiment_config["harness"]["chain_lengths"]

all_rows = []
perplexities = {}
for condition in CONDITIONS:
    print(f"=== Evaluating {condition} ===")
    rows, perplexity = evaluate_condition(condition, experiment_config, models_config, chain_lengths)
    all_rows.extend(rows)
    perplexities[condition] = perplexity
    print(f"{condition}: {len(rows)} tasks run, perplexity={perplexity:.2f}")

output = write_eval_outputs(all_rows, perplexities, REPO_ROOT / "results")
print("wrote results/eval_results.jsonl, eval_results.csv, eval_summary.json")

=== Evaluating base ===
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 3.257 GiB
no_split classes   : ['Qwen3DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 11.166 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.95 GiB  weights  1.760 GiB  free 11.192 GiB  reserve 11.166 GiB
  cuda:1  budget  13.00 GiB  weights  1.497 GiB  free 11.499 GiB  reserve

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Unsloth 2026.9.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https

KeyboardInterrupt: 

## Results

`output["summary"]` (one entry per condition x chain length) already has everything below — this just tabulates/plots it. See `evaluation/metrics.py`'s module docstring for exactly what each field means, especially the "DESIGN DECISION" section on how a step that succeeds after a retry is credited.

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(output["summary"])
summary_df[["condition", "chain_length", "n_tasks", "full_chain_success_rate",
            "per_step_success_rate", "clean_step_success_rate", "recovery_rate", "perplexity"]]

### Success rate vs chain length — the key figure for the research question

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for condition in CONDITIONS:
    sub = summary_df[summary_df["condition"] == condition].sort_values("chain_length")
    ax.plot(sub["chain_length"], sub["full_chain_success_rate"], marker="o", label=condition)
ax.set_xlabel("chain length")
ax.set_ylabel("full-chain success rate")
ax.set_xticks(chain_lengths)
ax.set_ylim(0, 1.05)
ax.set_title("Does multi-step success degrade faster than single-step?")
ax.legend()
plt.tight_layout()
plt.show()

### Error breakdown per condition (protocol vs tool-execution — `harness/errors.py`)

In [ ]:
from adbench.evaluation.metrics import error_category_breakdown

for condition in CONDITIONS:
    condition_rows = [r for r in all_rows if r["condition"] == condition]
    print(condition, error_category_breakdown(condition_rows))